<a href="https://colab.research.google.com/github/mahdifarsibaf/2-similar-DOE/blob/main/Acronym_Extractor_for_Thesis_PDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

upload the file

In [4]:
!pip install PyPDF2 python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 15.4 MB/s eta 0:00:00


In [1]:
from google.colab import files
uploaded = files.upload()

Saving PhD proposal V22 after prof carlos.pdf to PhD proposal V22 after prof carlos.pdf


In [7]:
import re
import PyPDF2
from docx import Document
from collections import OrderedDict

def extract_acronyms_from_pdf(pdf_path):
    """
    Extract acronyms and abbreviations from a PDF file.

    Args:
        pdf_path (str): Path to the PDF file

    Returns:
        dict: Dictionary of acronyms and their definitions
    """
    # Open the PDF file
    with open(pdf_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)

        acronyms = {}
        acronym_pattern = re.compile(r'\b([A-Z]{2,}s?)\b')
        definition_pattern = re.compile(r'([A-Z][A-Za-z\s]{5,})\s*\(([A-Z]{2,})\)')

        # Extract text from each page
        for page in pdf_reader.pages:
            text = page.extract_text()
            if not text:
                continue

            # Find acronym definitions (e.g., "Wire Arc Additive Manufacturing (WAAM)")
            definitions = definition_pattern.findall(text)
            for definition, acronym in definitions:
                acronyms[acronym] = definition.strip()

            # Find standalone acronyms (all caps words with 2+ letters)
            potential_acronyms = acronym_pattern.findall(text)
            for acronym in potential_acronyms:
                # Skip common non-acronyms
                if (len(acronym) > 5 or
                    acronym in ['THE', 'AND', 'FOR', 'THIS', 'THAT', 'WITH', 'FROM']):
                    continue
                if acronym not in acronyms:
                    acronyms[acronym] = "Definition not found"

    return acronyms

def create_acronym_doc(acronyms, output_path):
    """
    Create a Word document with the list of acronyms in alphabetical order.

    Args:
        acronyms (dict): Dictionary of acronyms and definitions
        output_path (str): Path for the output Word document
    """
    doc = Document()
    doc.add_heading('List of Abbreviations and Acronyms', 0)

    # Sort acronyms alphabetically
    sorted_acronyms = sorted(acronyms.items(), key=lambda x: x[0].lower())

    # Create a table with acronyms and definitions
    table = doc.add_table(rows=1, cols=2)
    table.style = 'Table Grid'

    # Set table headers
    hdr_cells = table.rows[0].cells
    hdr_cells[0].text = 'Acronym'
    hdr_cells[1].text = 'Definition'

    # Add acronyms to the table in alphabetical order
    for acronym, definition in sorted_acronyms:
        row_cells = table.add_row().cells
        row_cells[0].text = acronym
        row_cells[1].text = definition

    doc.save(output_path)

if __name__ == "__main__":
    # Specify your PDF file path
    pdf_file = "PhD proposal V22 after prof carlos.pdf"

    # Extract acronyms
    acronyms = extract_acronyms_from_pdf(pdf_file)

    # Create Word document
    output_file = "thesis_acronyms.docx"
    create_acronym_doc(acronyms, output_file)

    print(f"Extracted {len(acronyms)} acronyms. Saved to {output_file}")

Extracted 169 acronyms. Saved to thesis_acronyms.docx
